# Init

In [ ]:
# init
from importlib import reload
from pprint import pprint
import os
import sys
from pathlib import Path
import pandas as pd
import time
from IPython.display import clear_output

import tests.osm_test as osmt
import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsOSM.overpass as too
import toolsPandas.helpers as tph

def pckg_reload():
    reload(tgm)
    reload(too)
    reload(osmt)
    reload(tph)
    reload(tgl)

sys.path.append('..')
sys.path.append('../tools')
devPath = "C:/Users/gonta/D/software developer/"
sys.path.append(os.path.join(devPath, "packages"))

# Testing

## * import

In [1]:
data_dir = Path(os.path.join(os.getcwd(), '..', 'data/normalized'))
files = [f for f in data_dir.rglob('*.pkl')]
print(len(files))

df_by_cntr = {file.parent.name:tgm.load(str(file)) for file in files}
print(len(df_by_cntr))

71
71


In [2]:
df_all = pd.concat(df_by_cntr.values(), ignore_index=True)
print(len(df_all))

152991


In [3]:
id_tuples = df_all[['id','tags.parent_id','tags.country_id']].fillna('missing').apply(tuple,axis=1).to_list()
print(len(id_tuples))

id_tuples_df = pd.DataFrame(id_tuples, columns=['id','tags.parent_id','tags.country_id'])
print(len(id_tuples_df))

152991
152991


## 1. basic elems test

In [4]:
test_path = os.path.join('..', 'data', 'tests results', 'osm basic test')

In [5]:
logger = tgl.initiate_logger('logger')
save_logger_path = os.path.join('..', 'data', 'tests results', 'osm basic test', 'basic_test_res.log')
tgl.add_file_handler(logger, save_logger_path)

### * select to test

In [6]:
processsed = [f.name for f in Path(test_path).glob('*')]
print(len(processsed))
to_process = {k:v for k,v in df_by_cntr.items() if k not in processsed}
print(len(to_process))

72
1


### * make test

In [7]:
for country, df in to_process.items():
    logger.info(f'* country: {country}')
    test_res = osmt.osm_basic_test(df)
    tgm.dump(os.path.join(test_path, country, f'{country}_basic_test_res.pkl'), test_res)
    # time.sleep(2)
    clear_output(wait=True)

[INFO] : * country: UnitedStates
[INFO] :  * missing names: 4
[INFO] :  * relations from other countries: 351


### * review

In [8]:
test_res_by_cntr = {f.parent.name:tgm.load(f) for f in Path(test_path).glob('*/*')}
print(len(test_res_by_cntr))

71


In [11]:
temp = pd.DataFrame(test_res_by_cntr)
temp = temp.T.map(len)
temp['total'] = list(map(len, list(df_by_cntr.values())))
tph.df_peek(temp)

,missing.name,leak,in_country,NA_result,total
Afghanistan,0,0,24,11,35
Albania,0,0,390,0,390
Algeria,0,0,1,2147,2148
Andorra,0,0,1,0,1
Angola,0,1,1,62,64
Anguilla,0,0,1,0,1
AntiguaAndBarbuda,0,0,1,11,12
Argentina,1,1,885,325,1211
Armenia,0,1,26,0,27
Australia,0,0,1,615,616


### * derivate data

In [12]:
missing_names = []
leaks = []
for k,v in test_res_by_cntr.items():
    missing_names.extend(v['missing.name'])
for k,v in test_res_by_cntr.items():
    leaks.extend(v['leak'])

print(len(missing_names))
print(len(leaks))

22
1202


In [13]:

print(tgm.count_duplicates(missing_names))

{'18997125': 2}


In [14]:
print(tgm.count_duplicates(leaks))
print(tgm.count_duplicates([ele[0] for ele in leaks]))

{}
{'7400': 2, '3737915': 2, '3737930': 2, '3741133': 2, '3741188': 2, '3741189': 2, '3030949': 4, '3030951': 2, '4632921': 2, '4632922': 2, '4632923': 2, '4632926': 2, '4634580': 2, '4634581': 2, '4634582': 2, '4634583': 2, '4634584': 2, '4634585': 2, '4634586': 2, '4634587': 2, '4634588': 2, '4634589': 2, '4634590': 2, '4634591': 2, '4634592': 2, '4634594': 2, '4634595': 2, '4634596': 2, '4634597': 2, '4634598': 2, '4634600': 2, '4634613': 2, '8337869': 2, '4636132': 2, '4636133': 2, '4636134': 2, '4636135': 2, '4636136': 2, '4636137': 2, '4636138': 2, '4636139': 2, '4636140': 2, '4636141': 2, '4636142': 2, '4636143': 2, '4636144': 2, '4636145': 2, '4636146': 2, '4636147': 2, '4636148': 2, '4638811': 2, '4638812': 2, '4638813': 2, '4638814': 2, '4638815': 2, '4638816': 2, '4638817': 2, '4638818': 2, '4638819': 2, '4638820': 2, '4638821': 2, '4638822': 2, '4638823': 2, '4638824': 2, '4638825': 2, '4638826': 2, '4638827': 2, '4638828': 2, '8337854': 2, '4633166': 2, '4633168': 2, '4633

In [15]:
missing_names_tuples = [ele for ele in id_tuples if ele[0] in missing_names]
print(len(missing_names_tuples))
tgm.count_duplicates(missing_names_tuples)

22


{}

In [16]:
print(tgm.intersection(leaks, missing_names_tuples))

basic_to_delete = leaks + missing_names_tuples
print(len(basic_to_delete))

[]
1224


In [17]:
tgm.dump(os.path.join(test_path, "basic_to_delete.pkl"), basic_to_delete)

## 2. osm test if first level divisions belong to country

In [18]:
test_path = os.path.join('..', 'data', 'tests results', 'osm first level test')

### * select to test

In [19]:
print(len(df_by_cntr))
first_lvl_by_cntr = {}
for country, df in df_by_cntr.items():
    if not df[df['tags.admin_level'] == '4'].empty:
        first_lvl_by_cntr[country] = df[df['tags.admin_level'] == '4']
    else:
        print(country)
print(len(first_lvl_by_cntr))

71
Andorra
Anguilla
Estonia
FaroeIslands
Ireland
IsleOfMan
Latvia
Montenegro
NorthMacedonia
SanMarino
VaticanCity
60


In [20]:
first_lvl_df = df_all[df_all['tags.admin_level'] == '4']
print(len(first_lvl_df))
first_lvl_id_tuples = first_lvl_df[['id','tags.parent_id','tags.country_id']].apply(tuple,axis=1).to_list()
print(len(first_lvl_id_tuples))
tgm.count_duplicates(first_lvl_id_tuples)

1175
1175


{}

In [21]:
processed_tuples = tgm.load(os.path.join(test_path, "first_level_test_tuples_processed.pkl"))
print(len(processed_tuples))

1127


In [22]:
to_process = {}
for country,df in first_lvl_by_cntr.items():
    is_in_tuples_df = df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(processed_tuples)
    to_process_df = df[~is_in_tuples_df]
    if not to_process_df.empty:
        to_process[country] = to_process_df

print(len(to_process))
to_process.keys()

1


dict_keys(['UnitedStates'])

### * make tests

In [25]:
logger = tgl.initiate_logger('logger')
save_logger_path = os.path.join(test_path, 'first_level_test_res.log')
tgl.add_file_handler(logger, save_logger_path)
print(test_path)

..\data\tests results\osm first level test


In [27]:
for country, df in to_process.items():
    save_path = os.path.join(test_path, country, f"{country}_first_level_test_res.pkl")
    test_res_stored = tgm.load(save_path) if os.path.exists(save_path) else {}
    processed_tuples = test_res_stored.keys()

    to_test_df = df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(processed_tuples)
    to_test_df = df[~to_test_df]

    test_res = osmt.osm_test_center(
        to_test_df,
        save_temp=True,
        save_path=save_path
    )
    clear_output(wait=True)

[INFO] : * country: Armenia
[INFO] :  ^ [11/11]: testing ('3917110', '364066', '364066'):
[INFO] :   * saving ...
[INFO] :  $ finished: status: {'admin_centre': 'ok', 'label': 'missing', 'place': 'ok', 'geom_node': 'ok', 'centroid': 'ok'}


### * review

In [28]:
files = [f for f in Path(test_path).glob('*/*')]
print(len(files))
test_res_by_cntr = {}
for f in files:
    test_res_by_cntr[f.parent.name] = tgm.load(str(f))

print(len(test_res_by_cntr))

test_res_by_elem = {k:v for list in test_res_by_cntr.values() for k,v in list.items()}
print(len(test_res_by_elem))

59
59
1127


In [ ]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in test_res_by_elem.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                        1445
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]               154
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                         130
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                               22
[[admin_centre, missing], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                      6
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, missing], [centroid, missing]]       2
Name: count, dtype: int64

In [30]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         5138
missing     494
error         3
Name: count, dtype: int64

In [67]:
list(test_res_by_elem.keys())[list(result_status).index([['admin_centre', 'ok'], ['label', 'missing'], ['place', 'ok'], ['geom_node', 'ok'], ['centroid', 'ok']] )]

('7739254', '53292', '53292')

### * update processed

In [31]:
first_level_test_tuples_processed = [ele for list in list(test_res_by_cntr.values()) for ele in list]
print(len(first_level_test_tuples_processed))

1127


In [32]:
tgm.dump(os.path.join(test_path, "first_level_test_tuples_processed.pkl"), first_level_test_tuples_processed)

### * select false entities and childs to delete

In [33]:
test_res_simplified = {}
for id, val in test_res_by_elem.items():
    true_count = 0
    false_count = 0
    for type in ['admin_centre', 'label', 'place', 'geom_node', 'centroid']:
        if val[type]['status'] == 'ok':
            # select the first one that gives True
            # test_res_simplified[id] = False
            # if val[type]['result'] is True:
            #     test_res_simplified[id] = val[type]['result']
            #     break

            # assign the first valid result
            # test_res_simplified[id] = val[type]['result']
            # break

            # make a voting weight selection
            if val[type]['result'] is True: true_count += 1
            else: false_count += 1
    
    test_res_simplified[id] = true_count >= false_count


pd.Series([v for k,v in test_res_simplified.items()]).value_counts()

True     1105
False      22
Name: count, dtype: int64

In [34]:
tuples_to_delete_lvl1 = [k for k,v in test_res_simplified.items() if v is False]
print(len(tuples_to_delete_lvl1))
tuples_to_delete_lvl1

22


[('2279589', '195267', '195267'),
 ('301542', '286393', '286393'),
 ('364077', '286393', '286393'),
 ('364078', '286393', '286393'),
 ('364079', '286393', '286393'),
 ('364080', '286393', '286393'),
 ('364081', '286393', '286393'),
 ('364082', '286393', '286393'),
 ('364083', '286393', '286393'),
 ('364084', '286393', '286393'),
 ('364086', '286393', '286393'),
 ('364087', '286393', '286393'),
 ('3917110', '286393', '286393'),
 ('4217435', '52411', '52411'),
 ('405935', '59470', '59470'),
 ('153549', '167454', '167454'),
 ('47826', '51477', '51477'),
 ('223136', '192307', '192307'),
 ('184877', '184843', '184843'),
 ('1908810', '50371', '50371'),
 ('2060182', '192757', '192757'),
 ('4496067', '174737', '174737')]

In [73]:
df_all[(df_all[['id','tags.parent_id','tags.country_id']] == ('4496067', '174737', '174737')).all(axis=1)]

,type,id,tags.admin_level,tags.parent_id,tags.name,tags.name:en,tags.alt_name:en,tags.ISO3166-1,tags.ISO3166-2,tags.is_in:country,tags.ref:nuts,tags.ref:nuts:2,tags.ref:nuts:3,tags.addr:country,tags.country_name,tags.country_id
101218,relation,4496067,4,174737,Αποκεντρωμένη Διοίκηση Αιγαίου,Aegean,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Turkey,174737


In [35]:
temp = [[ele[0],ele[2]] for ele in tuples_to_delete_lvl1]
tuples_to_delete_childs = [ele for ele in id_tuples if [ele[1],ele[2]] in temp]
print(len(tuples_to_delete_childs))

tuples_to_delete = tuples_to_delete_childs + tuples_to_delete_lvl1
print(len(tuples_to_delete))

82
104


In [36]:
tgm.dump(os.path.join(test_path, "tuples_to_delete.pkl"), tuples_to_delete)

## 3. osm duplicates test

In [4]:
test_path = os.path.join('..', 'data', 'tests results', 'osm duplicates test')

### * check data

In [5]:
dups_id = df_all[df_all.duplicated(subset=['id'])]['id'].to_list()
print(len(dups_id))

dups_df = df_all[df_all.duplicated(subset=['id'], keep=False)]
print(len(dups_df))

dups_id_tuples = dups_df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).to_list()
print(len(dups_id_tuples))

1193
1805
1805


In [6]:
print('rows with duplicates (id):', len(id_tuples_df[id_tuples_df['id'].duplicated(keep=False)]))
print('unique duplicates (id):', len(id_tuples_df[id_tuples_df['id'].duplicated()]))

rows with duplicates (id): 1805
unique duplicates (id): 1193


In [7]:
print(
    'rows with duplicates (id, parent):',
    len(id_tuples_df[id_tuples_df[['id','tags.parent_id']].duplicated(keep=False)])
)
print(
    'rows with duplicates (id, parent_id, country_id):',
    len(id_tuples_df[id_tuples_df[['id','tags.parent_id','tags.country_id']].duplicated(keep=False)])
)

rows with duplicates (id, parent): 1410
rows with duplicates (id, parent_id, country_id): 0


In [9]:
to_review_ids = [('72639','60189','60189'), ('72639','60199','60199'),('6543282', '6543265', '192756'),
 ('6543282', '6543273', '192756'), ('59190', '59195', '59065'), ('59190', '59752', '59065')]

In [10]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "to_review_ids.pkl"), to_review_ids)

### [dev]

In [ ]:
temp1 = too.is_child_inside_parent('6543282','6543265')
[[k,v['result']] for k,v in temp1.items()]

[INFO] :    > Getting admin_centre:
[INFO] :     * Found node (admin_centre)
[INFO] :    > Testing admin_centre node (id: 7934379969)
[INFO] :     * Finished testing (admin_centre): False
[INFO] :    > Getting label:
[INFO] :     * Missing node (label)
[INFO] :    > Getting place:
[INFO] :     * Found node (place)
[INFO] :    > Testing place node (id: 7934379969)
[INFO] :     * Finished testing (place): False
[INFO] :    > Getting geom_node:
[INFO] :     * Found node (geom_node)
[INFO] :    > Testing geom_node node (id: 4298631214)
[INFO] :     * Finished testing (geom_node): True
[INFO] :    > Getting centroid:
[INFO] :     * Found node (centroid)
[INFO] :    > Testing centroid (lat: 36.8944071, lon: 6.4872707)
[INFO] :     * Finished testing (centroid): True


[['admin_centre', False],
 ['label', None],
 ['place', False],
 ['geom_node', True],
 ['centroid', True]]

In [ ]:
tph.get_from_df(df_all,['id','tags.parent_id','tags.country_id'],('6543282', '6543265', '192756'))

,type,id,tags.admin_level,tags.parent_id,tags.name,tags.name:us,tags.ISO3166-1,tags.ISO3166-2,tags.is_in:country,tags.ref:nuts,tags.ref:nuts:2,tags.ref:nuts:3,tags.addr:country,tags.country_name,tags.country_id
2462,rel,6543282,8,6543265,Beni Zid ⴱⴻⵏⵉ ⵣⵉⴷ بنى زيد,<NA>,<NA>,<NA>,Algeria,<NA>,<NA>,<NA>,<NA>,Algeria,192756


### * select to test

In [20]:
dups_test_processed = tgm.load(os.path.join(test_path, "dups_test_processed.pkl"))
print(len(dups_test_processed))

1759


In [21]:
to_process = {}
# there are duplicates between countries, so checking by country is not enough. Use list of all duplicates by id found
print('duplicates by id count:', len(dups_id_tuples))
print('id tuples processed:', len(dups_test_processed))

for country, df in df_by_cntr.items():
    df_id_tuples = df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1)
    to_process_bool_series = df_id_tuples.isin(dups_id_tuples) & ~df_id_tuples.isin(dups_test_processed)
    
    to_process_df = df[to_process_bool_series]
    if not to_process_df.empty:
        to_process[country] = to_process_df

to_process_all_tuples = [ele for df in to_process.values() for ele in df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1)]

print('countries to process: ', len(to_process))
print('id tuples to process: ', len(to_process_all_tuples))

duplicates by id count: 1805
id tuples processed: 1759
countries to process:  2
id tuples to process:  46


### * make tests

In [22]:
for country, df in to_process.items():
    save_path = os.path.join(test_path, country, f"{country}_dups_test_res.pkl")
    test_res_stored = tgm.load(save_path) if os.path.exists(save_path) else {}
    processed_tuples = test_res_stored.keys()

    to_test_df = df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(processed_tuples)
    to_test_df = df[~to_test_df]

    test_res = osmt.osm_test_center(
        to_test_df,
        save_temp=True,
        save_path=save_path
    )
    clear_output(wait=True)

[INFO] : * country: Chile
[INFO] :  ^ [15/15]: testing ('1980641', '1980749', '167454'):
[INFO] :   * saving ...
[INFO] :  $ finished: status: {'admin_centre': 'ok', 'label': 'missing', 'place': 'ok', 'geom_node': 'ok', 'centroid': 'ok'}


In [ ]:
('5320537', '9583346', '286393')

In [ ]:
[2, 1, 2, 1, 2]

### * review

In [23]:
files = [f for f in Path(test_path).glob('*/*')]
print(len(files))
test_res_by_cntr = {}
for f in files:
    test_res_by_cntr[f.parent.name] = tgm.load(str(f))

print(len(test_res_by_cntr))

test_res_by_elem = {k:v for list in test_res_by_cntr.values() for k,v in list.items()}
print(len(test_res_by_elem))

29
29
1805


In [24]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in test_res_by_elem.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                        1471
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]               154
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                         148
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                               24
[[admin_centre, missing], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                      6
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, missing], [centroid, missing]]       2
Name: count, dtype: int64

In [25]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         6922
missing    2103
Name: count, dtype: int64

### * update processed data

In [26]:
dups_test_processed = [ele for list in list(test_res_by_cntr.values()) for ele in list]
print(len(dups_test_processed))

1805


In [27]:
tgm.dump(os.path.join(test_path, "dups_test_processed.pkl"), dups_test_processed)

### * review

In [ ]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in test_res.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                        1399
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]               154
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                          55
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                               20
[[admin_centre, missing], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                      6
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, missing], [centroid, missing]]       2
Name: count, dtype: int64

In [ ]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         6242
missing    1938
Name: count, dtype: int64

### * select false entities and childs to delete

In [30]:
test_res_simplified = {}
for id, val in test_res_by_elem.items():
    true_count = 0
    false_count = 0
    for type in ['admin_centre', 'label', 'place', 'geom_node', 'centroid']:
        if val[type]['status'] == 'ok':
            # select the first one that gives True
            # test_res_simplified[id] = False
            # if val[type]['result'] is True:
            #     test_res_simplified[id] = val[type]['result']
            #     break

            # assign the first valid result
            # test_res_simplified[id] = val[type]['result']
            # break

            # make a voting weight selection
            if val[type]['result'] is True: true_count += 1
            else: false_count += 1
    
    test_res_simplified[id] = true_count >= false_count


pd.Series([v for k,v in test_res_simplified.items()]).value_counts()

True     1616
False     189
Name: count, dtype: int64

In [31]:
dups_tuples_to_delete_parents = [k for k,v in test_res_simplified.items() if v is False]
print(len(dups_tuples_to_delete_parents))

189


In [33]:
temp = [[ele[0],ele[2]] for ele in dups_tuples_to_delete_parents]
dups_tuples_to_delete_childs = [ele for ele in id_tuples if [ele[1],ele[2]] in temp]
print(len(dups_tuples_to_delete_childs))

# dups_tuples_to_delete = dups_tuples_to_delete_childs + dups_tuples_to_delete_parents
dups_tuples_to_delete = dups_tuples_to_delete_parents
print(len(dups_tuples_to_delete))

930
189


In [ ]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete.pkl"), dups_tuples_to_delete)

### * add manual review deletion

In [35]:
to_review_ids = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "to_review_ids.pkl"))
print(len(to_review_ids))

6


In [39]:
dups_tuples_to_delete_manual = [('6543282', '6543273', '192756'), ('59190', '59195', '59065')]

In [40]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete_manual.pkl"), dups_tuples_to_delete_manual)

In [41]:
dups_tuples_to_delete = dups_tuples_to_delete + dups_tuples_to_delete_manual
print(len(dups_tuples_to_delete))

191


In [42]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete.pkl"), dups_tuples_to_delete)

In [43]:
print(len(dups_id_tuples))
print(len(dups_tuples_to_delete))
temp = tgm.complement(dups_id_tuples, dups_tuples_to_delete)
print(len(temp))

1805
191
1614
